In [3]:
!pip install groq

In [4]:
!pip install groq
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

In [ ]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Hello"}]
)

print(response.choices[0].message.content)

Hello. How can I help you today?


In [ ]:
!pip install requests

In [ ]:
import requests

url = "https://www.themealdb.com/api.php"
url = "https://www.themealdb.com/api/json/v1/1/random.php"
url = "https://www.themealdb.com/api/json/v1/1/lookup.php?i=52772"


response = requests.get(url)
data = response.json()

print(data)

{'meals': [{'idMeal': '52772', 'strMeal': 'Teriyaki Chicken Casserole', 'strMealAlternate': None, 'strCategory': 'Chicken', 'strArea': 'Japanese', 'strInstructions': 'Preheat oven to 350° F. Spray a 9x13-inch baking pan with non-stick spray.\r\nCombine soy sauce, ½ cup water, brown sugar, ginger and garlic in a small saucepan and cover. Bring to a boil over medium heat. Remove lid and cook for one minute once boiling.\r\nMeanwhile, stir together the corn starch and 2 tablespoons of water in a separate dish until smooth. Once sauce is boiling, add mixture to the saucepan and stir to combine. Cook until the sauce starts to thicken then remove from heat.\r\nPlace the chicken breasts in the prepared pan. Pour one cup of the sauce over top of chicken. Place chicken in oven and bake 35 minutes or until cooked through. Remove from oven and shred chicken in the dish using two forks.\r\n*Meanwhile, steam or cook the vegetables according to package directions.\r\nAdd the cooked vegetables and ri

In [ ]:
if data["meals"] is None:
    print("No recipe found ❌")
else:
    meal = data["meals"][0]

In [ ]:
title = meal["strMeal"]
instructions = meal["strInstructions"]

ingredients = []

for i in range(1, 21):
    ing = meal[f"strIngredient{i}"]
    meas = meal[f"strMeasure{i}"]

    if ing and ing.strip():
        ingredients.append(f"{meas} {ing}")

In [ ]:
!pip install langchain faiss-cpu sentence-transformers

In [ ]:
!pip install faiss-cpu sentence-transformers
!pip install groq requests

In [ ]:
!pip install langchain langchain-community langchain-text-splitters
!pip install faiss-cpu sentence-transformers
!pip install groq requests

In [ ]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
def fetch_recipe(meal_name):
    url = f"https://www.themealdb.com/api/json/v1/1/search.php?s={meal_name}"
    response = requests.get(url)
    data = response.json()

    meals = data.get("meals")

    if not meals:
        print("No recipe found ❌")
        return None

    meal = meals[0]

    title = meal["strMeal"]

    ingredients = []
    for i in range(1, 21):
        ing = meal.get(f"strIngredient{i}")
        measure = meal.get(f"strMeasure{i}")
        if ing and ing.strip():
            ingredients.append(f"{measure} {ing}")

    instructions = meal["strInstructions"]

    return title, ingredients, instructions

In [ ]:
def create_recipe_text(title, ingredients, instructions):
    return f"""
Recipe: {title}

Ingredients:
{chr(10).join(ingredients)}

Instructions:
{instructions}
"""

In [ ]:
def create_vector_db(recipe_text):
    splitter = CharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=50
    )

    docs = splitter.create_documents([recipe_text])

    embeddings = HuggingFaceEmbeddings()

    db = FAISS.from_documents(docs, embeddings)

    return db

In [ ]:
def get_context(query, db):
    retriever = db.as_retriever()
    docs = retriever.invoke(query)

    context = " ".join([doc.page_content for doc in docs])
    return context

In [ ]:
chat_history = []

In [ ]:
def ask_ai(question, context, chat_history):

    history_text = ""
    for q, a in chat_history:
        history_text += f"User: {q}\nAI: {a}\n"

    prompt = f"""
You are an AI cooking assistant.

Conversation so far:
{history_text}

Context:
{context}

User Question:
{question}

Answer clearly and helpfully.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [ ]:
meal_name = input("Enter recipe name: ")

result = fetch_recipe(meal_name)

if result:
    title, ingredients, instructions = result

    recipe_text = create_recipe_text(title, ingredients, instructions)
    db = create_vector_db(recipe_text)

    chat_history = []

    print("\n✅ Conversational Cooking Assistant Ready! (type 'exit' to stop)\n")

    while True:
        question = input("You: ")

        if question.lower() == "exit":
            break

        context = get_context(question, db)

        answer = ask_ai(question, context, chat_history)

        print("AI:", answer)

        # 🔥 Save memory
        chat_history.append((question, answer))


Enter recipe name: salmon


/tmp/ipykernel_2396/414419257.py:9: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✅ Conversational Cooking Assistant Ready! (type 'exit' to stop)

You: i want to make salmon fry 
AI: To make salmon fry, you'll need a different recipe than the one we were working with, which was for Salmon Noodle Soup. 

Here's a simple recipe for Salmon Fry:

Ingredients:
- 2 Salmon fillets (you already have 2 salmon)
- 1/2 cup All-purpose flour
- 1/2 teaspoon Salt
- 1/4 teaspoon Black pepper
- 1/4 teaspoon Paprika
- 2 tablespoons Oil (e.g., olive or coconut oil)
- 2 lemons, cut into wedges (optional)
- Chopped coriander or parsley for garnish (you already have coriander)

Instructions:
1. Rinse the salmon fillets and pat them dry with a paper towel.
2. In a shallow dish, mix together the flour, salt, pepper, and paprika.
3. Dip each salmon fillet into the flour mixture, coating both sides evenly.
4. Heat the oil in a large non-stick skillet over medium-high heat.
5. Add the coated salmon fillets to the skillet and cook for 3-4 minutes per side, or until they're cooked through and 

KeyboardInterrupt: Interrupted by user